# MapBiomas Fogo – U-Net experimental – Brasil – MONITOR_01

**Alvo:** área queimada | **Sensor:** Sentinel-2 (10 m) | **Periodicidade:** mensal | Três blocos: **1 Configuração** → **2 Treino** → **3 Classificação**. Cada bloco é uma célula com abas; rode uma célula por vez, de cima para baixo.

## 📖 Guia rápido
Abra esta seção para ver o que cada bloco faz e onde ficam os resultados.

| Bloco | O que faz | Escolhe | Saída (GCS `.../MONITOR_01/UNET_01/{bioma}/{região}/training_…/`) |
|---|---|---|---|
| **1 Configuração** | login e escolha de **país → bioma → região → treino** (novo ou existente) | região, treino | `config.json` da versão |
| **2 Treino** | amostras e versão · parâmetros da U-Net · chips, estatísticas e treino · métricas | amostras, bandas, índices, parâmetros | `chips/`, `stats/`, `models/`, `reports/` |
| **3 Classificação** | modelo e meses · classificar (GPU) · juntar as faixas · subir ao GEE | modelo e meses | `predict/`, coleção no GEE |

**Mapa final:** 3 bandas int16 (nodata −9999): `burned` (1 = fogo no limiar da validação), `prob` (probabilidade × 1000) e `doy` (dia do ano).

**Se o Colab desconectar:** rode o bloco 1, confirme e abra o bloco 3 (ou 2); cada etapa grava o resultado no GCS e continua de onde parou.

## 1️⃣ Configuração
Login e escolha de país, bioma, região e treino. Clique em **Confirmar configuração**.

In [ ]:
# Configuração: prepara o Colab, faz o login e abre a escolha país → bioma → região → treino
import sys, os, subprocess, shutil

BRANCH = "classification-unet"
REPO_DIR = "/content/brazil-fire"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/mapbiomas/brazil-fire.git", REPO_DIR], check=True)
EXP_DIR = os.path.join(REPO_DIR, "network/catalog/experiments/unet_pantanal")
subprocess.run(["pip", "install", "-q", "-r", os.path.join(EXP_DIR, "requirements.txt")], check=True)
sys.path.insert(0, os.path.join(EXP_DIR, "src"))
for _m in [m for m in sys.modules if m.startswith("unet_")]:   # sem isto o Python usa a versão antiga
    del sys.modules[_m]

import unet_env
from unet_config import make_config

UCFG = make_config(
    # projeto (onde estão os dados do catálogo)
    COUNTRY="brazil", BUCKET="mapbiomas-fire", GCS_PREFIX="sudamerica/brazil/CATALOG_01", CAMPAIGN="MONITOR_01",
    GEE_PROJECT="mapbiomas-brazil",
    # GEE_PROJECT="mapbiomas-fire-485203",   # projeto que roda as tasks (GEE_EXPORT_PROJECT na config)
    # asset de regiões: region_name = bioma + "_r" + id (calculado, como no script do GEE)
    REGIONS_ASSET="users/geomapeamentoipam/AUXILIAR/regioes_biomas_col2",
    REGION_PROPERTY="region_name", REGION_NAME_FIELDS=["bioma", "id"], REGION_NAME_JOIN="_r",
    SENSOR="sentinel2", MOSAIC="minnbr",          # mosaicos: LIBRARY_IMAGES/SENTINEL2/MONTHLY/MINNBR/{período}/COG/
    EXPERIMENT="UNET_01",                          # pasta das saídas no GCS, dentro da campanha
    VALID_BAND=None,                               # None = sem dayOfYear; "dayOfYear" = usa a banda do mosaico
)
unet_env.connect(UCFG)                             # pop-up do Colab; GEE com a sua conta, GCS com a conta de serviço

from unet_app import abrir_config
APP = abrir_config(UCFG)                           # escolha país → bioma → região → treino e clique em Confirmar

## 2️⃣ Treino
Abas: **1 Amostras e versão** · **2 Parâmetros** · **3 Treinar** (conferir, chips, estatísticas, treino) · **4 Métricas** (cartões, gráficos e comparação com a MLP).

In [ ]:
from unet_app import abrir_treino

abrir_treino(APP)

## 3️⃣ Classificação
Parte de um treino já feito. Abas: **1 Modelo e meses** · **2 Classificar** (GPU, faixas salvas no GCS) · **3 Juntar** as faixas em um mapa (COG) · **4 Publicar no GEE**.

In [ ]:
from unet_app import abrir_classificacao

abrir_classificacao(APP)